## 1. Install TPU-Compatible JAX Stack
This notebook is rebuilt as a clean, ordered flow. We begin by installing a TPU-safe JAX stack appropriate for Kaggle TPU v3-8. We use JAX 0.4.34 TPU wheels and align NumPy and ml-dtypes. We defer MaxText deps to after clone.

Key goals:
- Ensure 8 TPU devices are accessible
- Avoid resolver upgrades that break TPU wheels


In [1]:
# 1) Install TPU-safe JAX stack
!pip install --no-deps --force-reinstall \
  "numpy==1.26.4" \
  "ml-dtypes==0.4.0" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jaxlib==0.4.34" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jax[tpu]==0.4.34" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html \
  --quiet

import jax
print("JAX:", jax.__version__, "TPU devices:", jax.device_count())



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


E0000 00:00:1757682450.045116      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:230


JAX: 0.4.34 TPU devices: 8


## 2. Clone MaxText at a Compatible Commit
Clone the MaxText repository and pin to a pre-pallas commit that does not require `jax.experimental.pallas.ops.attention`. We first try a Git-based search; if inconclusive, we fall back to manual scanning of recent commits.


# 2) Clone and pin MaxText
[Made Redundant by Step 7]

!git clone https://github.com/google/maxtext.git || true
%cd /kaggle/working/maxtext

import subprocess, os

# Try to find introduction commit for pallas.ops.attention and checkout its parent
patterns = [
    "pallas.ops.attention",
    "from jax.experimental.pallas.ops import attention",
]
culprit = None
for pattern in patterns:
    r = subprocess.run(['git', 'log', '-S', pattern, '--pretty=format:%H', '-n', '1'], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        culprit = r.stdout.strip().split('\n')[0]
        break

if culprit:
    print("First commit with pallas.ops.attention:", culprit)
    subprocess.run(['git', 'checkout', f'{culprit}^'], check=False)
    print(subprocess.check_output(['git', 'show', '-s', '--format=%ci %H', 'HEAD'], text=True))
else:
    # Fallback: pick a known pre-pallas date range (e.g., <= 2024-03-15) and pick that commit
    fallback = subprocess.check_output(['git', 'rev-list', '-n', '1', '--before=2024-03-15', 'HEAD'], text=True).strip()
    if fallback:
        print("Fallback commit (pre-2024-03-15):", fallback)
        subprocess.run(['git', 'checkout', fallback], check=False)
        print(subprocess.check_output(['git', 'show', '-s', '--format=%ci %H', 'HEAD'], text=True))
    else:
        print("Warning: could not determine a pre-pallas commit; staying on current HEAD.")

# Sanity: detect pallas import in current tree
has_pallas = False
try:
    with open('MaxText/layers/attentions.py', 'r') as f:
        has_pallas = 'pallas.ops.attention' in f.read()
except FileNotFoundError:
    pass
print("attentions.py uses pallas:", has_pallas)


# Find and remove all __pycache__ directories to prevent using stale code

[Made Redundant by Step 7]

!find . -type d -name "__pycache__" -exec rm -r {} +
print("✅ Python bytecode cache cleared successfully.")


## 3. Install MaxText Dependencies (Avoid Upgrading JAX)
Install MaxText requirements but protect the JAX pins by reapplying them immediately after. This balances repo requirements with TPU-safe versions.


# 3) Install MaxText deps, then re-pin JAX stack
%cd /kaggle/working/maxtext
!pip install -r requirements.txt --quiet || true

# Re-assert JAX pins to prevent resolver upgrades breaking TPU wheels
!pip install --no-deps --force-reinstall \
  "numpy==1.26.4" \
  "ml-dtypes==0.4.0" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jaxlib==0.4.34" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jax[tpu]==0.4.34" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html \
  --quiet
!pip install --no-deps --force-reinstall \
  "flax==0.10.4" \
  "optax==0.2.5" \
  "chex==0.1.89" \
  "orbax-checkpoint==0.11.5" \
  --quiet

import jax, flax, optax
print("JAX:", jax.__version__, "Flax:", flax.__version__, "Optax:", optax.__version__)


## 4. Verify TPU Devices and Basic JAX Runtime
Quick sanity check: confirm 8 TPU devices and a trivial JAX op run. This ensures runtime is consistent before loading MaxText.


In [2]:
# 4) TPU device and trivial op
import jax, jax.numpy as jnp
n = jax.device_count()
print(f"TPU devices: {n}")
print("Trivial JAX op:", jnp.add(1, 4))
if n != 8:
    print("⚠️ Warning: Expected 8 TPU cores. Verify accelerator is TPU v3-8.")


TPU devices: 8
Trivial JAX op: 5


## 5. Configure Kaggle Dataset Checkpoint Path
Set the path to the uploaded Kaggle dataset containing the MaxText Orbax checkpoint and validate key files exist.
- Accept either `<slug>/llama-3.1-8b-maxtext-checkpoint/` or directly `<slug>/` structures.


In [3]:
# 5) Determine checkpoint directory in Kaggle input
from pathlib import Path

DATASET_SLUG = "llama-3-1-8b-maxtext-checkpoint"  # change to your dataset slug if different
ROOT = Path("/kaggle/input") / DATASET_SLUG

inner = ROOT / "llama-3.1-8b-maxtext-checkpoint"
if (inner / "_CHECKPOINT_METADATA").exists() or (inner / "0").exists():
    base = inner
else:
    base = ROOT

# prefer step dir "0" if exists, else last numeric
step = None
if (base / "0").exists():
    step = base / "0"
else:
    nums = [p for p in base.iterdir() if p.is_dir() and p.name.isdigit()]
    if nums:
        step = sorted(nums, key=lambda p: int(p.name))[-1]

CKPT_DIR = step if step else base
print("Dataset root:", ROOT)
print("Checkpoint dir:", CKPT_DIR)

required = [
    CKPT_DIR / "_CHECKPOINT_METADATA",
    CKPT_DIR / "items" / "_METADATA",
]
for p in required:
    print("Exists", p, p.exists())

items_dir = CKPT_DIR / "items"
print("Items dir:", items_dir, items_dir.exists())


Dataset root: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
Checkpoint dir: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
Exists /kaggle/input/llama-3-1-8b-maxtext-checkpoint/_CHECKPOINT_METADATA True
Exists /kaggle/input/llama-3-1-8b-maxtext-checkpoint/items/_METADATA True
Items dir: /kaggle/input/llama-3-1-8b-maxtext-checkpoint/items True


## 6. Generate Minimal MaxText Config
Create a tiny YAML config with `load_parameters_path` pointing to the checkpoint and `steps: 1` for a minimal verification run.


In [4]:
# 6) Write minimal config YAML
import yaml
from pathlib import Path

CONFIG_DIR = Path("/kaggle/working/config")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / "minimal_maxtext_config.yaml"

cfg = {
    "run_name": "llama31_8b_verify",
    "load_parameters_path": str(CKPT_DIR),
    "steps": 1,
    "dataset_type": "none",
}

with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Config:")
print(CONFIG_PATH.read_text())


Config:
run_name: llama31_8b_verify
load_parameters_path: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
steps: 1
dataset_type: none



## 7. Install AQT (Pinned SHA) and tensorboardX, then verify imports
This step aligns AQT with the legacy MaxText commit and ensures required modules are importable before training.

- Pin AQT to a known-good 2023 commit SHA (`3275a461e59b90558352f1b40209e13462f44c38`).
- Install from the commit `.zip` (no git auth), with a fallback to `git clone` + `git checkout` + local install.
- Verify imports for `aqt.jax.v2.aqt_dot_general`, `aqt.jax.v2.google`, and `aqt.jax.v2.google.maxtext_sweeps`.
- Install `tensorboardX` in the same cell to avoid timing issues.
- Expected: `AQT ready: True` and `tensorboardX install exit code: 0`.

In [5]:
# 7) AQT setup (pinned commit only), install + verify, vendor v2 if missing, then shim google module if needed
%cd /kaggle/working

import sys, subprocess, importlib, pathlib, shutil

def try_import(name: str) -> bool:
    try:
        importlib.import_module(name)
        print('Import OK:', name)
        return True
    except Exception as e:
        print('Import failed:', name, e)
        return False

PINNED_SHA = '3275a461e59b90558352f1b40209e13462f44c38'  # 2023-09-07

print('Uninstalling any existing PyPI aqt package (Anki AQT)...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'aqt'])

print('Cloning google/aqt repository fresh...')
subprocess.run(['bash', '-lc', 'rm -rf /kaggle/working/aqt-src'])
subprocess.run(['bash', '-lc', 'git clone https://github.com/google/aqt.git /kaggle/working/aqt-src'])

print('Checking out pinned commit:', PINNED_SHA)
# Robust fetch then checkout to avoid "not our ref" with shallow clone
subprocess.run(['bash', '-lc', 'cd /kaggle/working/aqt-src && git fetch --unshallow || git fetch --all --tags --prune'])
subprocess.run(['bash', '-lc', f'cd /kaggle/working/aqt-src && git checkout {PINNED_SHA}'])

print('Installing google/aqt from local source (no deps)...')
install_ret = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--quiet', '/kaggle/working/aqt-src'])
print('pip install aqt-src exit code:', install_ret.returncode)

print('Installing tensorboardX...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'tensorboardX'])

base_ok = try_import('aqt')
v2_ok = try_import('aqt.jax.v2')
dot_ok = try_import('aqt.jax.v2.aqt_dot_general') if v2_ok else False

# Vendor v2 from source into site-packages if missing
if base_ok and not v2_ok:
    try:
        import aqt as _aqt
        src_dir = pathlib.Path('/kaggle/working/aqt-src/aqt/jax/v2')
        if src_dir.exists():
            dest_base = pathlib.Path(_aqt.__file__).parent / 'jax'
            dest_base.mkdir(parents=True, exist_ok=True)
            dest_dir = dest_base / 'v2'
            shutil.copytree(src_dir, dest_dir, dirs_exist_ok=True)
            print('Vendored AQT v2 from source into:', dest_dir)
            v2_ok = try_import('aqt.jax.v2')
            dot_ok = try_import('aqt.jax.v2.aqt_dot_general') if v2_ok else False
        else:
            print('Source v2 directory not found; cannot vendor.')
    except Exception as e:
        print('Vendoring v2 failed:', e)

sweeps_ok = try_import('aqt.jax.v2.google.maxtext_sweeps') if v2_ok else False

# If legacy google module missing but base aqt v2 is present, create a minimal shim
if v2_ok and not sweeps_ok:
    try:
        import aqt as _aqt
        base = pathlib.Path(_aqt.__file__).parent / 'jax' / 'v2' / 'google'
        base.mkdir(parents=True, exist_ok=True)
        init_path = base / '__init__.py'
        if not init_path.exists():
            init_path.write_text('')
        sweeps_path = base / 'maxtext_sweeps.py'
        if not sweeps_path.exists():
            sweeps_path.write_text('def get_sweep():\n    return {}\n')
        print('Shim written at:', base)
        sweeps_ok = try_import('aqt.jax.v2.google.maxtext_sweeps')
    except Exception as e:
        print('Failed to create shim:', e)

print('AQT ready:', bool(base_ok and v2_ok and dot_ok and sweeps_ok))

/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/kaggle/working
Uninstalling any existing PyPI aqt package (Anki AQT)...


Cloning google/aqt repository fresh...


Cloning into '/kaggle/working/aqt-src'...


Checking out pinned commit: 3275a461e59b90558352f1b40209e13462f44c38


...

Installing google/aqt from local source (no deps)...



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


pip install aqt-src exit code: 0
Installing tensorboardX...



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


Import OK: aqt
Import failed: aqt.jax.v2 No module named 'aqt.jax.v2'
Vendored AQT v2 from source into: /usr/local/lib/python3.10/site-packages/aqt/jax/v2
Import OK: aqt.jax.v2
Import OK: aqt.jax.v2.aqt_dot_general
Import failed: aqt.jax.v2.google.maxtext_sweeps No module named 'aqt.jax.v2.google'
Shim written at: /usr/local/lib/python3.10/site-packages/aqt/jax/v2/google
Import OK: aqt.jax.v2.google.maxtext_sweeps
AQT ready: True


## 8. Fresh clone MaxText, pin compatible commit, and run train in-process
This step ensures we run MaxText from a clean tree at a commit that avoids `jax.experimental.pallas` and `colocated_python`, with in-process execution to prevent TPU lock conflicts.

- Remove and re-clone `google/maxtext` into `/kaggle/working/maxtext`.
- Identify and checkout the latest commit without `jax.experimental.pallas` and `colocated_python` (expected `6ce556e1`, 2023-09-11).
- Verify absence of those imports via `grep`.
- With AQT and `tensorboardX` installed, execute `MaxText.train` in-process using `runpy`.
- Try common config flags in order: `--config`, `--config_path`, `--config_file`, `--config_files`, `--yaml_config`.
- Expected: one attempt returns code 0 for the minimal `steps: 1` verification.


In [6]:
# 8) Run MaxText from pinned commit (with deps + sys.path fix)
%cd /kaggle/working/

print("💣 Removing existing maxtext directory...")
!rm -rf maxtext

print("✨ Cloning a fresh copy of the repository...")
!git clone https://github.com/google/maxtext.git
%cd maxtext

MAXTEXT_COMMIT_HASH = "6ce556e1582cb37f4060c78fea3a62cc2be87fc2"  # 2023-09-11
print(f"Checking out known-good MaxText commit: {MAXTEXT_COMMIT_HASH}")
!git checkout {MAXTEXT_COMMIT_HASH}

# Provenance
!git rev-parse HEAD
!head -n 40 MaxText/train.py | sed -e 's/^/[train.py] /'

# Install minimal deps without breaking JAX pins
!pip install --quiet -r requirements.txt || true
!pip install --no-deps --force-reinstall --quiet "tensorflow-datasets==4.9.4" || true

print("\n🚀 Attempting in-process execution...")
import sys, runpy, os
from pathlib import Path

CONFIG_PATH = Path("/kaggle/working/config/minimal_maxtext_config.yaml")
repo_root = "/kaggle/working/maxtext"
pkg_root = "/kaggle/working/maxtext/MaxText"

# Ensure both the repo and package roots are importable
for p in (repo_root, pkg_root):
    if p not in sys.path:
        sys.path.insert(0, p)

entrypoint_module = "MaxText.train"
saved_argv = sys.argv[:]
try:
    import importlib

    # JAX KeyArray compatibility shim for older MaxText code
    import jax, jax.numpy as jnp
    if not hasattr(jax.random, "KeyArray"):
        try:
            jax.random.KeyArray = jax.Array  # type: ignore[attr-defined]
        except Exception:
            jax.random.KeyArray = jnp.ndarray  # type: ignore[assignment]

    # Try multiple config flag variants sequentially
    flag_variants = [
        '--config',
        '--config_file',
        '--config_files',
        '--yaml_config',
        '--config_path',
    ]
    success = False
    import traceback
    for flag in flag_variants:
        try:
            sys.argv = ['-m', entrypoint_module, f'{flag}={CONFIG_PATH}']
            print(f"Executing ({flag}): python3 -m {entrypoint_module} {flag}={CONFIG_PATH}")
            runpy.run_module(entrypoint_module, run_name='__main__')
            print(f"✅✅✅ SUCCESS ({flag})")
            success = True
            break
        except SystemExit as e:
            # absl.app may call sys.exit on flag parsing error; treat non-zero as failure
            if getattr(e, 'code', 1) != 0:
                print(f"Flag {flag} failed with SystemExit code {e.code}")
            else:
                print(f"Exited cleanly with SystemExit 0 for {flag}")
                success = True
                break
        except Exception:
            print(f"❌ Execution failed for {flag}. Traceback:\n{traceback.format_exc()}")
    if not success:
        # Show helpshort to reveal valid flags
        try:
            sys.argv = ['-m', entrypoint_module, '--helpshort']
            print('Fetching helpshort to discover valid flags...')
            runpy.run_module(entrypoint_module, run_name='__main__')
        except Exception:
            pass
        raise RuntimeError('All config flag variants failed. See logs above for details.')
except Exception:
    import traceback
    print(f"❌ Execution failed. Traceback:\n{traceback.format_exc()}")
finally:
    sys.argv = saved_argv

/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/kaggle/working
💣 Removing existing maxtext directory...


/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


✨ Cloning a fresh copy of the repository...
Cloning into 'maxtext'...
remote: Enumerating objects: 56157, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (220/220), done.
remote: Total 56157 (delta 240), reused 194 (delta 156), pack-reused 55774 (from 2)
Receiving objects: 100% (56157/56157), 317.56 MiB | 29.65 MiB/s, done.
Resolving deltas: 100% (41575/41575), done.
/kaggle/working/maxtext
Checking out known-good MaxText commit: 6ce556e1582cb37f4060c78fea3a62cc2be87fc2
Note: switching to '6ce556e1582cb37f4060c78fea3a62cc2be87fc2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:


/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-12 13:08:58.454578: I tensorflow/core/tpu/tpu_api_dlsym_initializer.cc:95] Opening library: /usr/local/lib/python3.10/site-packages/tensorflow/python/platform/../../libtensorflow_cc.so.2
2025-09-12 13:08:58.454832: I tensorflow/core/tpu/tpu_api_dlsym_initializer.cc:121] Libtpu path is: /usr/local/lib/python3.10/site-packages/libtpu/libtpu.so
2025-09-12 13:08:58.460801: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/kaggle/working/maxtext/MaxText/train.py:60: DeprecationWarning: initialize_cache i

...

Found 8 devices.


/kaggle/working/maxtext/MaxText/train.py:60: DeprecationWarning: initialize_cache is deprecated; use set_cache_dir instead
  cc.initialize_cache(os.path.expanduser("~/jax_cache"))
FATAL Flags parsing error: Unknown command line flag 'yaml_config'
Pass --helpshort or --helpfull to see help on flags.
/kaggle/working/maxtext/MaxText/train.py:60: DeprecationWarning: initialize_cache is deprecated; use set_cache_dir instead
  cc.initialize_cache(os.path.expanduser("~/jax_cache"))
FATAL Flags parsing error: Unknown command line flag 'config_path'
Pass --helpshort or --helpfull to see help on flags.
/kaggle/working/maxtext/MaxText/train.py:60: DeprecationWarning: initialize_cache is deprecated; use set_cache_dir instead
  cc.initialize_cache(os.path.expanduser("~/jax_cache"))


Flag --yaml_config failed with SystemExit code 1
Executing (--config_path): python3 -m MaxText.train --config_path=/kaggle/working/config/minimal_maxtext_config.yaml
Found 8 devices.
Flag --config_path failed with SystemExit code 1
Fetching helpshort to discover valid flags...
Found 8 devices.
Automatically created module for IPython interactive environment

Try --helpfull to get a list of all flags.


SystemExit: 1

/usr/local/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
